# r2bulk_dg — round 2 from vh3: DANGER-amplified: same corpus, w=1+6*danger (P90 state ~3.8x, P99 ~6.6x)

All vectors train FROM vh3 (20k bar: 13,765 / 9,658). The ENDPOINT is a direction
generator — expect it to lose; the candidate is vh3 + a*(endpoint-vh3) merged locally.
Recipe: pure hard-CE (blend 0.0), bs 32768, lr 3e-4, normal BN, per-epoch Drive saves.

**Upload to `MyDrive/alphatrain/`:**
1. `r2_bulk.pt.gz` (740,453,101 B)
2. `r2_bulk_era.npz` (12,513,259 B — NEW)
3. `small128_vh3.pt` (12,067,702 B — NEW)
3. tarball v5 — already on Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time
DRIVE='/content/drive/MyDrive/alphatrain'
!cp {DRIVE}/colorlines_pillar3d_v5.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v5.tar.gz
os.makedirs('/content/alphatrain/data', exist_ok=True)
t0=time.time()
!cp {DRIVE}/r2_bulk.pt.gz /content/r2_bulk.pt.gz
gz=os.path.getsize('/content/r2_bulk.pt.gz'); print(f'.gz: {gz:,} bytes')
assert gz == 740_453_101, f'.gz truncated! got {gz}; re-upload r2_bulk.pt.gz'
!gunzip -t /content/r2_bulk.pt.gz && echo '.gz integrity OK'
!gzip -dc /content/r2_bulk.pt.gz > /content/alphatrain/data/r2_bulk.pt
pt=os.path.getsize('/content/alphatrain/data/r2_bulk.pt')
assert pt == 2_039_621_645, f'.pt size wrong! got {pt}'
print(f'corpus: {pt/1e9:.2f} GB, 6,175,263 states ({time.time()-t0:.0f}s)')
!rm /content/r2_bulk.pt.gz
!cp {DRIVE}/small128_vh3.pt /content/alphatrain/data/
assert os.path.getsize('/content/alphatrain/data/small128_vh3.pt') == 12_067_702
!pip install -q numpy numba scipy


In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g=torch.cuda.get_device_properties(0); print(f'GPU {torch.cuda.get_device_name(0)} | {g.total_memory/1e9:.0f} GB')

In [ ]:
# ===== CONFIG (r2bulk_dg) =====
CHANNELS = 128
EPOCHS   = 12
BATCH    = 32768
LR       = 3e-4
T        = 1.0
DW       = 0
BLEND    = 0.0
GAMMA    = 1  # mask carries era*danger; trainer w = 1 + mask
SEED     = 42
SAVE_STEPS = 250
RUN      = "r2bulk_dg"
print(f'RUN={RUN} ep={EPOCHS} bs={BATCH} lr={LR} gamma={GAMMA}')


In [ ]:
# === Build this arm's per-row weights: era emphasis x danger (review #10 amendments) ===
import numpy as np, torch
d = torch.load('alphatrain/data/r2_bulk.pt', map_location='cpu', weights_only=False)
fresh = np.load('/content/drive/MyDrive/alphatrain/r2_bulk_era.npz')['fresh']
danger = d['disagree_mask'].numpy()  # continuous danger from the vh3 head
ERA_MULT = 2.1   # fresh-era rows (24.1%) -> ~40% of gradient mass
GAMMA_D = 6.0 if RUN.endswith('dg') else 0.0
w = np.where(fresh, ERA_MULT, 1.0) * (1.0 + GAMMA_D * danger)
d['disagree_mask'] = torch.from_numpy((w - 1.0).astype(np.float32))
torch.save(d, 'alphatrain/data/r2_bulk.pt')
print(f'weights set: mean {w.mean():.3f}, P90 {np.percentile(w,90):.2f}, P99 {np.percentile(w,99):.2f}')


In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/r2_bulk.pt \
    --resume alphatrain/data/small128_vh3.pt --warm-start \
    --channels {CHANNELS} --seed {SEED} --amp --compile \
    --epochs {EPOCHS} --batch-size {BATCH} --lr {LR} --warmup-epochs 1 \
    --target-temperature {T} --decisiveness-power {DW} --blend-alpha {BLEND} --disagree-gamma {GAMMA} \
    --save-every-steps {SAVE_STEPS} \
    --copy-to /content/drive/MyDrive/alphatrain/{RUN}_best.pt \
    --save-dir /content/drive/MyDrive/alphatrain/{RUN}_ckpts 2>&1 | tee /content/{RUN}_train.log
# val is informational only — the gate is GAMEPLAY (floor-first, 5k decides).


In [ ]:
# Checkpoints already save DIRECTLY to Drive ({RUN}_ckpts/) —
# every epoch_N.pt and e{E}_s{S}.pt lands there as it is written.
import glob
for f in sorted(glob.glob(f'/content/drive/MyDrive/alphatrain/{RUN}_ckpts/*.pt')):
    print(f)


## Gate protocol (M5, C++ eval — download checkpoints as they save)

```bash
python -m alphatrain.inference_cpp.export_ts --model alphatrain/data/iter5a_e1_s400.pt
mv alphatrain/inference_cpp/data/policy_ts.pt alphatrain/inference_cpp/data/dagger1_e1_s400_ts.pt
cd alphatrain/inference_cpp
./build/eval --model data/dagger1_e1_s400_ts.pt --device mps --seed-start 775000 --seed-end 775500 --batch 500
```
1. **Screen step checkpoints** (s100..s900 + epoch saves) at 500 seeds. The 500-seed screen is a
   **catastrophe filter ONLY** — it rejects blowups; it does NOT rank close calls (vh1 itself read
   P50 −6% at 500 seeds, then WON +4.6% at 5k — HISTORY:3375).
2. Take the 2-3 best-looking survivors (floor-first: <1000%, P5/P10) to the **5k eval**
   (`--seed-end 780000`) against vh1's bar: mean 13,080 / P50 9,323 / P5 1,222 / <1000 3.5%.
3. Expected per the MICRO-GO calibration: +3-8%% median. If it clears → name it `small128_vh2`,
   retrain the survival head on ITS backbone, re-export, then RE-GATE Engine B (own-MCTS judge)
   at the new level — the alternation plan (memory: project_small_model_distill).
4. All step checkpoints regress at 500 (catastrophically) → recheck mix %% and label sanity
   before ANY recipe surgery (HISTORY 178 lesson: measure, don't theorize).